In [5]:
import tlsh
import ssdeep
import os
from collections import defaultdict

In [10]:
def get_all_files(directory):
    """주어진 디렉토리 내 모든 파일을 재귀적으로 검색하고 파일명 기준으로 매핑"""
    file_dict = {}
    for root, _, files in os.walk(directory):
        for file in files:
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, directory)  # Seed 내부 상대경로 유지|

            if '|' in relative_path:
                relative_path = relative_path.split('|',1)[0]+'.exe'

            file_dict[relative_path] = full_path
    return file_dict

def compute_ssdeep_hash(file_path):
    """파일의 ssdeep 해시 값을 계산"""
    try:
        with open(file_path, "rb") as f:
            return ssdeep.hash(f.read())
    except Exception as e:
        print(f"오류 발생 (ssdeep): {file_path} - {e}")
        return None

def compute_tlsh_hash(file_path):
    """파일의 TLSH 해시 값을 계산"""
    try:
        with open(file_path, "rb") as f:
            return tlsh.hash(f.read())
    except Exception as e:
        print(f"오류 발생 (tlsh): {file_path} - {e}")
        return None

def compare_files(seed_files, target_files):
    """Seed 파일과 숫자 디렉토리 내 동일한 파일명만 비교"""
    pert_dict = defaultdict(list)
    for relative_path, seed_path in seed_files.items():
        if relative_path in target_files:
            target_path = target_files[relative_path]

            seed_ssdeep = compute_ssdeep_hash(seed_path)
            seed_tlsh = compute_tlsh_hash(seed_path)

            target_ssdeep = compute_ssdeep_hash(target_path)
            target_tlsh = compute_tlsh_hash(target_path)

            if not (seed_ssdeep and seed_tlsh and target_ssdeep and target_tlsh):
                continue

            ssdeep_similarity = ssdeep.compare(seed_ssdeep, target_ssdeep)
            tlsh_distance = tlsh.diff(seed_tlsh, target_tlsh)
           # prnit()

#             perturbation = target_path.split('|',1)[-1].replace('.exe','')
#             #print(perturbation, tlsh_distance, ssdeep_similarity)
#             pert_dict[perturbation].append((tlsh_distance,ssdeep_similarity))
            print(f"\n[파일 비교] {relative_path}")
            print(f"  - Seed 파일: {seed_path}")
            print(f"  - 대상 파일: {target_path}")
            print(f"    🔹 ssdeep 유사도: {ssdeep_similarity}%")
            print(f"    🔹 tlsh 거리: {tlsh_distance}")
            
#     #print(pert_dict)
#     for perturbation, sim_score in pert_dict.items():
#         #tlsh_score = sim_score[0]
#         #ssdeep_score  = sim_score[1]
#         total_sample = len(sim_score)
#         total_tlsh = 0
#         total_ssdeep = 0
#         for tlsh_score, ssdeep_score in sim_score:
#             total_tlsh += tlsh_score
#             total_ssdeep += ssdeep_score

#         total_tlsh  = total_tlsh/total_sample
#         total_ssdeep  = total_ssdeep/total_sample
#         print(perturbation, round(total_tlsh),round(total_ssdeep), total_sample)

if __name__ == "__main__":
    base_dir = "./input/"
    #seed_dir = os.path.join(base_dir, "Seed")
    number_dirs = "./m_sample_2/"

    seed_files = get_all_files(base_dir)
    target_files={}
    target_files.update(get_all_files(number_dirs))
    compare_files(seed_files, target_files)


[파일 비교] iexplore_32.exe
  - Seed 파일: ./input/iexplore_32.exe
  - 대상 파일: ./m_sample_2/iexplore_32|section_append.exe
    🔹 ssdeep 유사도: 97%
    🔹 tlsh 거리: 2

[파일 비교] putty.exe
  - Seed 파일: ./input/putty.exe
  - 대상 파일: ./m_sample_2/putty|section_append.exe
    🔹 ssdeep 유사도: 97%
    🔹 tlsh 거리: 1

[파일 비교] hello_world.exe
  - Seed 파일: ./input/hello_world.exe
  - 대상 파일: ./m_sample_2/hello_world|section_append.exe
    🔹 ssdeep 유사도: 91%
    🔹 tlsh 거리: 27
